# Foundational Probability Distributions for Quantitative Finance & Risk
### Interactive Visual Lab (Companion to `Quant_Risk_PDFs_Theory.tex`)

**Included Core Distributions (Filtered for the 4-Month Quant Risk Roadmap):**
1. **Bernoulli** $(p)$ — Binary events (Single-day VaR breach / Single-loan default)
2. **Binomial** $(n, p)$ — Exception counts (250-day Basel VaR backtesting & traffic-light zones)
3. **Continuous Uniform** $U(a, b)$ — Monte Carlo Inverse-Transform sampling & $p$-values
4. **Normal** $\mathcal{N}(\mu, \sigma^2)$ **& CLT** — Bell curve, 68-95-99.7 rule, and Central Limit Theorem
5. **Lognormal** $(\mu, \sigma^2)$ — Compounded asset prices & Black-Scholes GBM terminal price
6. **Exponential** $(\lambda)$ — Constant hazard rate, survival probability, and credit $\text{PD}(t)$
7. **Gamma** $(k, \theta)$ — Sum of waiting times & parent of $\chi^2$, Student's $t$, and Beta
8. **Chi-Square** $\chi^2(k)$ — Sum of squared normals, variance estimation, Jarque-Bera & LR tests
9. **Student's** $t(\nu)$ — Fat-tailed asset returns (scale mixture of normals) vs. Gaussian tails
10. **Fisher's** $F(d_1, d_2)$ — Ratio of variance estimates & OLS factor regression significance
11. **Beta** $(\alpha, \beta)$ — Bounded $[0,1]$ recovery rates / LGD & Bayesian PD updating
12. **Generalized Pareto** $\text{GPD}(\xi, \beta)$ — Extreme Value Theory (Peaks-Over-Threshold VaR & ES)
13. **Multivariate Normal vs. Multivariate Student's** $t$ — Cholesky simulation, $\mathbf{w}^\top\boldsymbol{\Sigma}\mathbf{w}$, and crash clustering

In [1]:
# ---------- SETUP: Run this cell first ----------
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
from ipywidgets import interact

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (12, 4),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10.5
})

def stats_str(d):
    """Return mean, variance, skewness, and excess kurtosis of a frozen scipy dist."""
    m, v, s, k = d.stats(moments='mvsk')
    f = lambda x: 'undef' if not np.isfinite(float(x)) else f'{float(x):.3f}'
    return f'mean={f(m)} | var={f(v)} | skew={f(s)} | ex.kurt={f(k)}'

def stem_plot(ax, ks, pmf, color='C0', label=None):
    ax.vlines(ks, 0, pmf, color=color, lw=3.5, alpha=0.75)
    ax.plot(ks, pmf, 'o', color=color, label=label)

print('Setup complete.')

Setup complete.


---
## 1. Bernoulli$(p)$: Single Binary Trial (VaR Hit or Loan Default)
$$P(X=x) = p^x(1-p)^{1-x}, \quad x \in \{0,1\} \qquad \mathbb{E}[X]=p, \quad \operatorname{Var}(X)=p(1-p)$$

In [2]:
def plot_bernoulli(p=0.10):
    d = st.bernoulli(p)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    ax[0].bar([0, 1], [1 - p, p], color=['#2980b9', '#c0392b'], width=0.45, edgecolor='k')
    ax[0].set_xticks([0, 1])
    ax[0].set_xticklabels(['0 (No Event)', '1 (VaR Breach / Default)'])
    ax[0].set_ylim(0, 1.05)
    ax[0].set_title(f'Bernoulli(p={p:.2f})  |  {stats_str(d)}')
    
    pp = np.linspace(0.001, 0.999, 300)
    ax[1].plot(pp, pp * (1 - pp), 'k-', lw=2, label='Var(X) = p(1-p)')
    ax[1].plot(p, p * (1 - p), 'ro', ms=9, label=f'Current Var = {p*(1-p):.4f}')
    ax[1].set_title('Variance Parabola (Maximized at p = 0.50)')
    ax[1].set_xlabel('Probability p')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

interact(plot_bernoulli, p=(0.01, 0.99, 0.01));

interactive(children=(FloatSlider(value=0.1, description='p', max=0.99, min=0.01, step=0.01), Output()), _dom_…

---
## 2. Binomial$(n, p)$: Number of Breaches in $n$ Independent Trials (Week 4 Basel Backtest)
$$P(X=k) = \binom{n}{k}p^k(1-p)^{n-k} \qquad \mathbb{E}[X]=np, \quad \operatorname{Var}(X)=np(1-p)$$

In [3]:
def plot_binomial_basel(n=250, p=0.01, observed_breaches=5, show_normal=True):
    d = st.binom(n, p)
    k_max = int(max(20, d.ppf(0.9995) + 3))
    ks = np.arange(0, k_max + 1)
    
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    stem_plot(ax[0], ks, d.pmf(ks), 'C0', 'Binomial PMF')
    if show_normal:
        x_grid = np.linspace(0, k_max, 400)
        ax[0].plot(x_grid, st.norm(n * p, np.sqrt(n * p * (1 - p))).pdf(x_grid), 'r--', lw=2, label='Normal Approx')
    ax[0].set_title(f'Binomial(n={n}, p={p:.3f})  |  {stats_str(d)}')
    ax[0].set_xlabel('Count k')
    ax[0].legend()
    
    # Basel Traffic Light & Kupiec LR_uc
    zones = ['green' if k <= 4 else ('gold' if k <= 9 else 'crimson') for k in ks]
    ax[1].bar(ks, d.pmf(ks), color=zones, edgecolor='k', alpha=0.8)
    ax[1].axvline(observed_breaches, color='navy', ls='--', lw=2.5, label=f'Observed = {observed_breaches}')
    p_hat = observed_breaches / n
    if 0 < observed_breaches < n:
        lr_uc = -2 * ((n - observed_breaches)*np.log((1 - p)/(1 - p_hat)) + observed_breaches*np.log(p/p_hat))
    else:
        lr_uc = 0.0
    pval = st.chi2.sf(lr_uc, df=1)
    ax[1].set_title(f'Basel Zones | Kupiec LR_uc = {lr_uc:.2f} (p-val = {pval:.4f})')
    ax[1].set_xlabel('Number of VaR Exceptions')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

interact(plot_binomial_basel, n=(50, 500, 10), p=(0.005, 0.15, 0.005), observed_breaches=(0, 20, 1));

interactive(children=(IntSlider(value=250, description='n', max=500, min=50, step=10), FloatSlider(value=0.01,…

---
## 3. Continuous Uniform $U(a, b)$ & Inverse-Transform Sampling (Monte Carlo Engine)
$$f(x) = \frac{1}{b-a}, \quad x \in [a,b] \qquad \mathbb{E}[X]=\frac{a+b}{2}, \quad \operatorname{Var}(X)=\frac{(b-a)^2}{12}, \quad \text{Ex.Kurt}=-1.2$$

In [4]:
def plot_uniform_mc(a=0.0, b=1.0, target_dist='Normal(0,1)', N_sim=5000):
    if b <= a + 0.1:
        b = a + 0.1
    d = st.uniform(loc=a, scale=b - a)
    x = np.linspace(a - 0.5*(b-a), b + 0.5*(b-a), 500)
    
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.9))
    ax[0].plot(x, d.pdf(x), 'k-', lw=2)
    ax[0].fill_between(x, d.pdf(x), alpha=0.25, color='C0')
    ax[0].set_title(f'Uniform({a:.1f}, {b:.1f})  |  {stats_str(d)}')
    ax[0].set_xlabel('x')
    
    # Inverse Transform Sampling X = F^{-1}(U)
    U = np.random.default_rng(7).uniform(0, 1, N_sim)
    if target_dist == 'Normal(0,1)':
        X_sim = st.norm.ppf(U)
        grid = np.linspace(-4, 4, 300)
        true_pdf = st.norm.pdf(grid)
    elif target_dist == 'Exponential(lam=1.5)':
        X_sim = -np.log(1.0 - U) / 1.5
        grid = np.linspace(0, 6, 300)
        true_pdf = st.expon(scale=1/1.5).pdf(grid)
    else:
        X_sim = st.t(df=4).ppf(U)
        grid = np.linspace(-6, 6, 300)
        true_pdf = st.t(df=4).pdf(grid)
        
    ax[1].hist(X_sim, bins=55, density=True, alpha=0.45, color='teal', label='Simulated F^(-1)(U)')
    ax[1].plot(grid, true_pdf, 'r-', lw=2.2, label=f'Exact {target_dist} PDF')
    ax[1].set_xlim(grid[0], grid[-1])
    ax[1].set_title(f'Inverse Transform Sampling (N={N_sim})')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

interact(plot_uniform_mc, a=(-3.0, 2.0, 0.2), b=(0.5, 5.0, 0.2),
         target_dist=['Normal(0,1)', 'Exponential(lam=1.5)', 'Student-t(df=4)'], N_sim=(500, 20000, 500));

interactive(children=(FloatSlider(value=0.0, description='a', max=2.0, min=-3.0, step=0.2), FloatSlider(value=…

---
## 4. Normal $\mathcal{N}(\mu, \sigma^2)$, Parametric VaR/CVaR, and the Central Limit Theorem
$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right) \qquad M_X(t) = \exp\left(\mu t + \frac{1}{2}\sigma^2 t^2\right)$$

In [5]:
def plot_normal_and_clt(mu=0.0, sigma=1.0, alpha=0.99, clt_n=5, clt_source='Exponential'):
    d = st.norm(mu, sigma)
    x = np.linspace(mu - 4.5*sigma, mu + 4.5*sigma, 800)
    
    fig, ax = plt.subplots(1, 2, figsize=(13.5, 4))
    ax[0].plot(x, d.pdf(x), 'k-', lw=2)
    var_q = d.ppf(alpha)
    es_q = mu + sigma * (st.norm.pdf(st.norm.ppf(alpha)) / (1.0 - alpha))
    ax[0].fill_between(x[x >= var_q], d.pdf(x[x >= var_q]), color='crimson', alpha=0.4,
                       label=f'Tail (1-alpha={1-alpha:.1%}): VaR={var_q:.2f}, ES={es_q:.2f}')
    ax[0].axvline(es_q, color='darkred', ls='--', lw=2, label=f'Expected Shortfall = {es_q:.2f}')
    ax[0].set_title(f'Normal({mu:.1f}, {sigma**2:.2f})  |  {stats_str(d)}')
    ax[0].legend(fontsize=9)
    
    # CLT Simulation
    rng = np.random.default_rng(12)
    N_draws = 15000
    if clt_source == 'Exponential':
        raw = rng.exponential(1.0, size=(N_draws, clt_n))
    elif clt_source == 'Uniform(0,1)':
        raw = rng.random(size=(N_draws, clt_n))
    else:
        raw = (rng.random(size=(N_draws, clt_n)) < 0.15).astype(float)
    sums = raw.sum(axis=1)
    z_sums = (sums - sums.mean()) / sums.std()
    z_grid = np.linspace(-4, 4, 300)
    ax[1].hist(z_sums, bins=60, density=True, range=(-4, 4), alpha=0.45, color='C0',
               label=f'Sum of n={clt_n} {clt_source}s')
    ax[1].plot(z_grid, st.norm.pdf(z_grid), 'r-', lw=2.2, label='Standard Normal N(0,1)')
    ax[1].set_title(f'CLT Convergence: Sample Skew={st.skew(z_sums):.2f}, Ex.Kurt={st.kurtosis(z_sums):.2f}')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

interact(plot_normal_and_clt, mu=(-3.0, 3.0, 0.2), sigma=(0.3, 3.0, 0.1),
         alpha=(0.90, 0.995, 0.005), clt_n=(1, 100, 1),
         clt_source=['Exponential', 'Uniform(0,1)', 'Bernoulli(0.15)']);

interactive(children=(FloatSlider(value=0.0, description='mu', max=3.0, min=-3.0, step=0.2), FloatSlider(value…

---
## 5. Lognormal$(\mu, \sigma^2)$ & Black-Scholes Geometric Brownian Motion (Week 11)
$$f(x) = \frac{1}{x\sigma\sqrt{2\pi}}\exp\left(-\frac{(\ln x - \mu)^2}{2\sigma^2}\right), \quad x > 0 \qquad \text{Mode } (e^{\mu-\sigma^2}) < \text{Median } (e^\mu) < \text{Mean } (e^{\mu+\sigma^2/2})$$

In [6]:
def plot_lognormal_gbm(S0=100.0, r=0.06, sigma=0.30, T=1.0):
    mu_ln = np.log(S0) + (r - 0.5 * sigma**2) * T
    sig_ln = sigma * np.sqrt(T)
    d = st.lognorm(s=sig_ln, scale=np.exp(mu_ln))
    x = np.linspace(max(0.5, d.ppf(0.001)), d.ppf(0.995), 600)
    
    mode_val = np.exp(mu_ln - sig_ln**2)
    med_val = np.exp(mu_ln)
    mean_val = np.exp(mu_ln + 0.5 * sig_ln**2)
    
    fig, ax = plt.subplots(1, 2, figsize=(13.5, 4))
    ax[0].plot(x, d.pdf(x), 'k-', lw=2)
    ax[0].axvline(mode_val, color='g', ls=':', lw=2, label=f'Mode = {mode_val:.2f}')
    ax[0].axvline(med_val, color='b', ls='--', lw=2, label=f'Median = {med_val:.2f}')
    ax[0].axvline(mean_val, color='r', lw=2, label=f'Mean S0*e^(rT) = {mean_val:.2f}')
    ax[0].set_title(f'GBM Terminal Price S_T  |  {stats_str(d)}')
    ax[0].set_xlabel('Stock Price S_T')
    ax[0].legend()
    
    # Log-space is Normal
    log_x = np.log(x)
    ax[1].plot(log_x, st.norm(mu_ln, sig_ln).pdf(log_x), 'purple', lw=2)
    ax[1].axvline(mu_ln, color='b', ls='--', label=f'E[ln S_T] = {mu_ln:.3f}')
    ax[1].set_title('In Log-Space: ln(S_T) is Symmetric Normal')
    ax[1].set_xlabel('ln(S_T)')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

interact(plot_lognormal_gbm, S0=(50, 200, 10), r=(0.0, 0.15, 0.01), sigma=(0.05, 0.80, 0.05), T=(0.25, 3.0, 0.25));

interactive(children=(IntSlider(value=100, description='S0', max=200, min=50, step=10), FloatSlider(value=0.06…

---
## 6. Exponential$(\lambda)$: Waiting Times & Constant Hazard Credit Default
$$f(t) = \lambda e^{-\lambda t}, \quad t \ge 0 \qquad S(t) = e^{-\lambda t}, \quad \text{PD}(t) = 1 - e^{-\lambda t}, \quad \mathbb{E}[T] = \frac{1}{\lambda}, \quad \operatorname{Var}(T) = \frac{1}{\lambda^2}$$

In [7]:
def plot_exponential_credit(lam=0.25, horizon_t=3.0):
    d = st.expon(scale=1.0 / lam)
    t_grid = np.linspace(0, max(10.0, d.ppf(0.995)), 500)
    
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.9))
    ax[0].plot(t_grid, d.pdf(t_grid), 'k-', lw=2, label='f(t) = lam * exp(-lam*t)')
    mask = t_grid <= horizon_t
    ax[0].fill_between(t_grid[mask], d.pdf(t_grid[mask]), color='crimson', alpha=0.35,
                       label=f'Cumulative PD({horizon_t:.1f}y) = {d.cdf(horizon_t):.2%}')
    ax[0].set_title(f'Exponential(lambda={lam:.2f})  |  {stats_str(d)}')
    ax[0].set_xlabel('Time t (Years)')
    ax[0].legend()
    
    ax[1].plot(t_grid, d.sf(t_grid), 'g-', lw=2.2, label='Survival S(t) = exp(-lam*t)')
    ax[1].plot(t_grid, d.cdf(t_grid), 'r--', lw=2.2, label='Cumulative PD(t) = 1 - exp(-lam*t)')
    ax[1].axvline(horizon_t, color='k', ls=':')
    ax[1].set_title(f'Constant Hazard Rate h(t) = {lam:.2f} / year')
    ax[1].set_xlabel('Time t (Years)')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

interact(plot_exponential_credit, lam=(0.05, 1.50, 0.05), horizon_t=(0.5, 10.0, 0.5));

interactive(children=(FloatSlider(value=0.25, description='lam', max=1.5, min=0.05, step=0.05), FloatSlider(va…

---
## 7. Gamma$(k, \theta)$: Waiting Time to $k$-th Event & Parent of $\chi^2$
$$f(x) = \frac{x^{k-1}e^{-x/\theta}}{\Gamma(k)\theta^k}, \quad x > 0 \qquad \mathbb{E}[X]=k\theta, \quad \operatorname{Var}(X)=k\theta^2, \quad \text{Skew}=\frac{2}{\sqrt{k}}, \quad \text{Ex.Kurt}=\frac{6}{k}$$

In [8]:
def plot_gamma_dist(k=3.0, theta=1.0, show_normal_limit=True):
    d = st.gamma(a=k, scale=theta)
    x = np.linspace(0.001, d.ppf(0.999), 600)
    
    fig, ax = plt.subplots(figsize=(11.5, 4))
    ax.plot(x, d.pdf(x), 'k-', lw=2.2, label=f'Gamma(k={k:.1f}, theta={theta:.1f})')
    ax.plot(x, st.expon(scale=theta).pdf(x), 'g:', lw=1.8, label='Exponential (k=1)')
    if show_normal_limit:
        ax.plot(x, st.norm(k * theta, np.sqrt(k) * theta).pdf(x), 'r--', lw=1.8,
                label='Normal Approx N(k*theta, k*theta^2)')
    ax.set_title(f'Gamma Distribution  |  {stats_str(d)}')
    ax.set_xlabel('x')
    ax.legend()
    plt.tight_layout()
    plt.show()

interact(plot_gamma_dist, k=(0.5, 25.0, 0.5), theta=(0.2, 3.0, 0.2));

interactive(children=(FloatSlider(value=3.0, description='k', max=25.0, min=0.5, step=0.5), FloatSlider(value=…

---
## 8. Chi-Square $\chi^2(k) = \text{Gamma}(k/2, 2)$: Sum of $k$ Squared Standard Normals
$$Q = \sum_{i=1}^k Z_i^2 \sim \chi^2(k) \qquad f(x) = \frac{x^{k/2-1}e^{-x/2}}{2^{k/2}\Gamma(k/2)} \qquad \mathbb{E}[Q]=k, \quad \operatorname{Var}(Q)=2k$$

In [ ]:
def plot_chi2_dist(k=2, alpha_crit=0.95, N_sim=20000):
    d = st.chi2(df=k)
    x = np.linspace(0.01, max(15, d.ppf(0.999)), 600)
    sim_q = (np.random.default_rng(5).standard_normal((N_sim, k))**2).sum(axis=1)
    crit_val = d.ppf(alpha_crit)
    
    fig, ax = plt.subplots(figsize=(11.5, 4))
    ax.hist(sim_q, bins=70, density=True, alpha=0.35, color='C0', label='Simulated Sum of k Squared Normals')
    ax.plot(x, d.pdf(x), 'k-', lw=2.2, label=f'Exact Chi2(df={k}) PDF')
    ax.fill_between(x[x >= crit_val], d.pdf(x[x >= crit_val]), color='crimson', alpha=0.45,
                    label=f'{alpha_crit:.0%} Critical Value = {crit_val:.2f} (Reject H0)')
    ax.set_xlim(0, x[-1])
    if k == 1:
        ax.set_ylim(0, 1.0)
    ax.set_title(f'Chi-Square(k={k})  |  {stats_str(d)}  (Note: k=1 is Kupiec/Christoffersen, k=2 is Jarque-Bera)')
    ax.legend()
    plt.tight_layout()
    plt.show()

interact(plot_chi2_dist, k=(1, 30, 1), alpha_crit=(0.90, 0.99, 0.01), N_sim=(5000, 50000, 5000));

interactive(children=(IntSlider(value=2, description='k', max=30, min=1), FloatSlider(value=0.95, description=…

---
## 9. Student's $t(\nu)$: Fat-Tailed Asset Returns via Scale Mixture of Normals
$$T = \frac{Z}{\sqrt{V/\nu}} \qquad f(t) = \frac{\Gamma(\frac{\nu+1}{2})}{\sqrt{\nu\pi}\,\Gamma(\frac{\nu}{2})}\left(1+\frac{t^2}{\nu}\right)^{-\frac{\nu+1}{2}} \qquad \operatorname{Var}(T)=\frac{\nu}{\nu-2} \ (\nu>2), \quad \text{Ex.Kurt}=\frac{6}{\nu-4} \ (\nu>4)$$

In [10]:
def plot_student_t(nu=4.0):
    d = st.t(df=nu)
    x = np.linspace(-8, 8, 800)
    sig_equiv = np.sqrt(nu / (nu - 2.0)) if nu > 2.0 else 1.0
    norm_matched = st.norm(0, sig_equiv)
    
    fig, ax = plt.subplots(1, 2, figsize=(13.5, 4))
    ax[0].plot(x, d.pdf(x), 'k-', lw=2.2, label=f'Student-t(nu={nu:.1f})')
    ax[0].plot(x, norm_matched.pdf(x), 'r--', lw=2, label='Normal (Matched Variance)')
    ax[0].set_title(f'Linear Scale PDF  |  {stats_str(d)}')
    ax[0].legend()
    
    x_tail = np.linspace(1.5, 12, 400)
    ax[1].semilogy(x_tail, d.pdf(x_tail), 'k-', lw=2.2, label='Student-t Power-Law Tail')
    ax[1].semilogy(x_tail, norm_matched.pdf(x_tail), 'r--', lw=2, label='Normal Exponential Tail')
    ax[1].set_ylim(1e-10, 1.0)
    ax[1].set_title('Right Tail on Log Scale (Fat Tails Dominate Beyond 3 Sigma)')
    ax[1].set_xlabel('t')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

interact(plot_student_t, nu=(1.5, 30.0, 0.5));

interactive(children=(FloatSlider(value=4.0, description='nu', max=30.0, min=1.5, step=0.5), Output()), _dom_c…

---
## 10. Fisher's $F(d_1, d_2)$: Ratio of Variance Estimates & OLS Factor Significance (Week 7)
$$F = \frac{U_1 / d_1}{U_2 / d_2} \qquad \mathbb{E}[F] = \frac{d_2}{d_2 - 2} \quad (d_2 > 2)$$

In [11]:
def plot_f_dist(d1_factors=4, d2_resid_df=120):
    d = st.f(d1_factors, d2_resid_df)
    x = np.linspace(0.01, max(5.0, d.ppf(0.998)), 600)
    crit95 = d.ppf(0.95)
    
    fig, ax = plt.subplots(figsize=(11.5, 4))
    ax.plot(x, d.pdf(x), 'k-', lw=2.2, label=f'F(d1={d1_factors}, d2={d2_resid_df})')
    ax.fill_between(x[x >= crit95], d.pdf(x[x >= crit95]), color='crimson', alpha=0.4,
                    label=f'95% Critical F = {crit95:.2f} (Factors Jointly Significant)')
    ax.set_title(f'F-Distribution  |  {stats_str(d)}')
    ax.set_xlabel('F-Statistic')
    ax.legend()
    plt.tight_layout()
    plt.show()

interact(plot_f_dist, d1_factors=(1, 20, 1), d2_resid_df=(10, 250, 5));

interactive(children=(IntSlider(value=4, description='d1_factors', max=20, min=1), IntSlider(value=120, descri…

---
## 11. Beta$(\alpha, \beta)$: Credit Loss Given Default (LGD $\in [0,1]$) & Bayesian Updating
$$f(x) = \frac{x^{\alpha-1}(1-x)^{\beta-1}}{B(\alpha,\beta)}, \quad x \in (0,1) \qquad \mathbb{E}[X]=\frac{\alpha}{\alpha+\beta}, \quad \operatorname{Var}(X)=\frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}$$

In [12]:
def plot_beta_credit(alpha=2.5, beta=3.5, obs_defaults=0, obs_loans=0):
    prior = st.beta(alpha, beta)
    x = np.linspace(0.002, 0.998, 600)
    
    fig, ax = plt.subplots(figsize=(11.5, 4))
    ax.plot(x, prior.pdf(x), 'b-', lw=2.2, label=f'Beta({alpha:.1f}, {beta:.1f}) | Mean={alpha/(alpha+beta):.2%}')
    ax.fill_between(x, prior.pdf(x), color='royalblue', alpha=0.2)
    
    if obs_loans > 0 and obs_defaults <= obs_loans:
        post_a = alpha + obs_defaults
        post_b = beta + (obs_loans - obs_defaults)
        post = st.beta(post_a, post_b)
        ax.plot(x, post.pdf(x), 'r--', lw=2.2,
                label=f'Posterior Beta({post_a:.1f}, {post_b:.1f}) | Mean={post_a/(post_a+post_b):.2%}')
    ax.set_title(f'Beta Distribution on [0, 1]  |  {stats_str(prior)}')
    ax.set_xlabel('Proportion / LGD / Default Probability')
    ax.legend()
    plt.tight_layout()
    plt.show()

interact(plot_beta_credit, alpha=(0.5, 15.0, 0.5), beta=(0.5, 15.0, 0.5),
         obs_defaults=(0, 30, 1), obs_loans=(0, 50, 5));

interactive(children=(FloatSlider(value=2.5, description='alpha', max=15.0, min=0.5, step=0.5), FloatSlider(va…

---
## 12. Generalized Pareto Distribution $\text{GPD}(\xi, \beta)$: Extreme Value Theory (Week 2 POT VaR & ES)
$$g_{\xi,\beta}(y) = \frac{1}{\beta}\left(1+\frac{\xi y}{\beta}\right)^{-1/\xi-1} \qquad \mathrm{VaR}_q = u + \frac{\beta}{\xi}\left(\left[\frac{N}{N_u}(1-q)\right]^{-\xi}-1\right), \qquad \mathrm{ES}_q = \frac{\mathrm{VaR}_q + \beta - \xi u}{1-\xi}$$

In [13]:
def plot_gpd_pot(nu_true=3.5, u_quantile=0.94, q_var=0.99, N_days=2500):
    rng = np.random.default_rng(42)
    scale_t = np.sqrt((nu_true - 2.0) / nu_true)
    losses = rng.standard_t(nu_true, size=N_days) * scale_t
    
    u = np.quantile(losses, u_quantile)
    excesses = losses[losses > u] - u
    Nu = len(excesses)
    xi_hat, _, beta_hat = st.genpareto.fit(excesses, floc=0)
    
    # POT VaR and ES vs Normal VaR and ES
    var_gpd = u + (beta_hat / xi_hat) * (((N_days / Nu) * (1.0 - q_var))**(-xi_hat) - 1.0)
    es_gpd = (var_gpd + beta_hat - xi_hat * u) / (1.0 - xi_hat) if xi_hat < 1.0 else np.nan
    
    mu_n, sig_n = losses.mean(), losses.std(ddof=1)
    z_q = st.norm.ppf(q_var)
    var_norm = mu_n + sig_n * z_q
    es_norm = mu_n + sig_n * (st.norm.pdf(z_q) / (1.0 - q_var))
    
    fig, ax = plt.subplots(1, 2, figsize=(13.5, 4))
    y_grid = np.linspace(0, excesses.max(), 300)
    ax[0].hist(excesses, bins=30, density=True, alpha=0.45, color='C0', label=f'{Nu} Excesses over u={u:.2f}')
    ax[0].plot(y_grid, st.genpareto(xi_hat, scale=beta_hat).pdf(y_grid), 'r-', lw=2.2,
               label=f'GPD Fit: xi={xi_hat:.2f}, beta={beta_hat:.2f}')
    ax[0].set_title('Peaks-Over-Threshold GPD Fit to Tail Losses')
    ax[0].set_xlabel('Excess Loss (Y = L - u)')
    ax[0].legend()
    
    x_idx = np.arange(2)
    ax[1].bar(x_idx - 0.18, [var_norm, var_gpd], width=0.35, color='#2980b9', label=f'VaR ({q_var:.1%})')
    ax[1].bar(x_idx + 0.18, [es_norm, es_gpd], width=0.35, color='#c0392b', label=f'ES ({q_var:.1%})')
    ax[1].set_xticks(x_idx)
    ax[1].set_xticklabels(['Normal Fit (Whole Series)', 'EVT-GPD Fit (Tail Only)'])
    gap_pct = (var_gpd - var_norm) / var_gpd * 100.0
    ax[1].set_title(f'Normal VaR Underestimates EVT Tail Risk by {gap_pct:.1f}%')
    ax[1].set_ylabel('Loss Magnitude')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

interact(plot_gpd_pot, nu_true=(2.5, 10.0, 0.5), u_quantile=(0.90, 0.97, 0.01), q_var=(0.98, 0.999, 0.001), N_days=(1000, 10000, 500));

interactive(children=(FloatSlider(value=3.5, description='nu_true', max=10.0, min=2.5, step=0.5), FloatSlider(…

---
## 13. Multivariate Normal $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ vs. Multivariate Student-$t$ (Week 6 Portfolio Risk)
$$\mathbf{X} = \boldsymbol{\mu} + \mathbf{L}\mathbf{Z} \quad (\boldsymbol{\Sigma}=\mathbf{L}\mathbf{L}^\top), \qquad \sigma_p^2 = \mathbf{w}^\top\boldsymbol{\Sigma}\mathbf{w}, \qquad \text{RC}_i = \frac{w_i(\boldsymbol{\Sigma}\mathbf{w})_i}{\sigma_p}$$

In [14]:
def plot_mvn_vs_mvt(rho=0.60, nu=3.5, w1=0.50, sigma1=1.0, sigma2=1.0, N_sim=3500):
    rng = np.random.default_rng(9)
    S = np.array([[sigma1**2, rho * sigma1 * sigma2], [rho * sigma1 * sigma2, sigma2**2]])
    L = np.linalg.cholesky(S)
    Z = rng.standard_normal((N_sim, 2))
    X_mvn = Z @ L.T
    
    V = rng.chisquare(nu, size=N_sim) / nu
    X_mvt = ((Z / np.sqrt(V)[:, None]) @ L.T) * np.sqrt((nu - 2.0) / nu)
    
    thr = -2.5
    p_crash_mvn = np.mean((X_mvn[:, 0] < thr*sigma1) & (X_mvn[:, 1] < thr*sigma2)) * 100.0
    p_crash_mvt = np.mean((X_mvt[:, 0] < thr*sigma1) & (X_mvt[:, 1] < thr*sigma2)) * 100.0
    
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    lim = 5.0 * max(sigma1, sigma2)
    ax[0].scatter(X_mvn[:, 0], X_mvn[:, 1], s=4, alpha=0.35, color='C0')
    ax[0].axvline(thr*sigma1, color='r', ls=':'); ax[0].axhline(thr*sigma2, color='r', ls=':')
    ax[0].set_xlim(-lim, lim); ax[0].set_ylim(-lim, lim)
    ax[0].set_title(f'Bivariate Normal (rho={rho:.2f})\nJoint < -2.5s Crash = {p_crash_mvn:.2f}%')
    
    ax[1].scatter(X_mvt[:, 0], X_mvt[:, 1], s=4, alpha=0.35, color='C1')
    ax[1].axvline(thr*sigma1, color='r', ls=':'); ax[1].axhline(thr*sigma2, color='r', ls=':')
    ax[1].set_xlim(-lim, lim); ax[1].set_ylim(-lim, lim)
    ax[1].set_title(f'Bivariate Student-t (nu={nu:.1f}, same Cov)\nJoint < -2.5s Crash = {p_crash_mvt:.2f}%')
    
    ws = np.linspace(0, 1, 101)
    vols = [np.sqrt(np.array([w, 1-w]) @ S @ np.array([w, 1-w])) for w in ws]
    w_vec = np.array([w1, 1 - w1])
    cur_vol = np.sqrt(w_vec @ S @ w_vec)
    ax[2].plot(ws, vols, 'k-', lw=2, label='Portfolio Volatility sqrt(w^T S w)')
    ax[2].plot(ws, ws*sigma1 + (1-ws)*sigma2, 'g--', label='Weighted Avg Vol (rho=1)')
    ax[2].plot(w1, cur_vol, 'ro', ms=8, label=f'Current Vol = {cur_vol:.3f}')
    ax[2].set_title('Diversification Benefit')
    ax[2].set_xlabel('Weight w1 in Asset 1')
    ax[2].legend(fontsize=8.5)
    plt.tight_layout()
    plt.show()

interact(plot_mvn_vs_mvt, rho=(-0.90, 0.90, 0.05), nu=(2.5, 25.0, 0.5),
         w1=(0.0, 1.0, 0.05), sigma1=(0.5, 2.0, 0.1), sigma2=(0.5, 2.0, 0.1), N_sim=(1000, 8000, 500));

interactive(children=(FloatSlider(value=0.6, description='rho', max=0.9, min=-0.9, step=0.05), FloatSlider(val…